# ACTIVA M1 — Step 4 gate v3: sub-window frame offset sweep

**Settings:** GPU T4, Internet ON. Same 4 videos as v1/v2.

v1/v2: every frame pipeline gives feature cosine 0.937–0.941 vs the published ActionFormer features,
with best *whole-window* shift 0. This run tests whether the original extraction started its windows a few
frames earlier/later: frames are decoded once (production ffmpeg path), then shifted by k frames at 30 fps
(k > 0 drops the first k frames, k < 0 repeats the first frame k times) before windowing.
No annotations are used. Output: `/kaggle/working/m1_step4_gate_v3.json`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ffmpeg -version | head -1
import sys, torch, numpy; print(sys.version.split()[0], "torch", torch.__version__, "numpy", numpy.__version__)

In [ ]:
%cd /kaggle/working
!rm -rf ACTIVA dl
!git clone -q -b m1-real-action-inference https://github.com/harshjha15335/ACTIVA.git && git -C ACTIVA log --oneline -1
!git clone -q https://github.com/epic-kitchens/epic-kitchens-download-scripts.git dl
!pip install -q fvcore iopath simplejson av gdown pytest
import sys; sys.path[:0] = ["/kaggle/working/ACTIVA/code"]

In [ ]:
!cd /kaggle/working/ACTIVA && python scripts/setup_m1_assets.py --dest /tmp/m1 > /kaggle/working/setup_m1.log 2>&1; grep -A1 '"SlowFast.pyth"' -m1 /kaggle/working/setup_m1.log
VIDEOS = ["P02_13", "P03_21", "P26_30", "P28_23"]
!cd /kaggle/working/dl && python epic_downloader.py --videos --specific-videos {",".join(VIDEOS)} --output-path /tmp/epic > /kaggle/working/dl.log 2>&1; tail -1 /kaggle/working/dl.log
import glob
MP4 = {v: glob.glob(f"/tmp/epic/**/{v}.MP4", recursive=True)[0] for v in VIDEOS}
print(MP4)

In [ ]:
import os, subprocess, numpy as np
TAR = "/tmp/epic_kitchens.tar.gz"
if not os.path.exists(TAR):
    subprocess.run(["gdown", "--fuzzy", "https://drive.google.com/file/d/1Z4U_dLuu6_cV5NBIrSzsSDOOj2Uar85X/view", "-O", TAR], check=True)
os.makedirs("/tmp/pub", exist_ok=True)
subprocess.run(["tar", "-xzf", TAR, "-C", "/tmp/pub", "--wildcards"] + [f"*/{v}.npz" for v in VIDEOS], check=True)
PUB = {}
for v in VIDEOS:
    f = np.load(glob.glob(f"/tmp/pub/**/{v}.npz", recursive=True)[0]); PUB[v] = {k: f[k] for k in f.files}
    print(v, {k: x.shape for k, x in PUB[v].items()})

In [ ]:
import time, json, pathlib, numpy as np, torch
import slowfast_features as sf, actions, video_io
from slowfast_features import Preprocess, FrameSource
from action_config import EngineAssets
A = EngineAssets(pathlib.Path("/tmp/m1"))
bundle = sf.load_slowfast(A.slowfast_code, A.slowfast_cfg, A.slowfast_ckpt, device="cuda")
P = Preprocess(color="bgr")
FR = {}
for v in VIDEOS:
    t0 = time.time(); FR[v] = list(sf.ffmpeg_frames(MP4[v], P, FrameSource())); print(v, len(FR[v]), f"{time.time()-t0:.1f}s")

def cos_rows(a, b):
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12); b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return (a * b).sum(1)

def shifted(frames, k):
    return frames[k:] if k >= 0 else [frames[0]] * (-k) + frames

OFFSETS = [-12, -8, -6, -4, -2, -1, 0, 1, 2, 4, 6, 8, 12]
REPORT = {}
for k in OFFSETS:
    for v in VIDEOS:
        out = sf.extract_features(None, bundle, P, batch_size=8, frames=iter(shifted(FR[v], k)))
        pub = PUB[v]; T = min(len(out["feats"]), len(pub["feats"]))
        c = cos_rows(out["feats"][:T], pub["feats"][:T])
        REPORT.setdefault(k, {})[v] = {"cos_mean": float(c.mean()), "cos_min": float(c.min()),
            "verb": float((out["v_prob"][:T].argmax(1) == pub["v_prob"][:T].argmax(1)).mean()),
            "noun": float((out["n_prob"][:T].argmax(1) == pub["n_prob"][:T].argmax(1)).mean()), "T": [len(out["feats"]), len(pub["feats"])]}
    m = {kk: float(np.mean([r[kk] for r in REPORT[k].values()])) for kk in ("cos_mean", "cos_min", "verb", "noun")}
    print(f"offset {k:+3d}: cos {m['cos_mean']:.4f} (min {m['cos_min']:.3f})  verb {m['verb']:.3f}  noun {m['noun']:.3f}  "
          + "  ".join(f"{v}:{REPORT[k][v]['cos_mean']:.3f}" for v in VIDEOS))

In [ ]:
SUMMARY = {k: {kk: float(np.mean([r[kk] for r in vids.values()])) for kk in ("cos_mean", "cos_min", "verb", "noun")} for k, vids in REPORT.items()}
best = max(SUMMARY, key=lambda k: SUMMARY[k]["cos_mean"]); print("best offset:", best, SUMMARY[best], " offset 0:", SUMMARY[0])
json.dump({"summary": {str(k): s for k, s in SUMMARY.items()}, "per_video": {str(k): r for k, r in REPORT.items()}, "best": best},
          open("/kaggle/working/m1_step4_gate_v3.json", "w"), indent=1)
print("saved")